# Download Dataset


In [2]:
!pip install aicrowd-cli

     |████████████████████████████████| 51kB 2.5MB/s 
     |████████████████████████████████| 215kB 7.6MB/s 
     |████████████████████████████████| 61kB 6.3MB/s 
     |████████████████████████████████| 81kB 8.4MB/s 
     |████████████████████████████████| 163kB 39.3MB/s 
     |████████████████████████████████| 61kB 6.9MB/s 
     |████████████████████████████████| 51kB 5.8MB/s 
     |████████████████████████████████| 71kB 7.6MB/s 
ERROR: google-colab 1.0.0 has requirement requests~=2.23.0, but you'll have requests 2.25.1 which is incompatible.
ERROR: datascience 0.10.6 has requirement folium==0.2.1, but you'll have folium 0.8.3 which is incompatible.
  Found existing installation: requests 2.23.0
    Uninstalling requests-2.23.0:
      Successfully uninstalled requests-2.23.0
  Found existing installation: tqdm 4.41.1
    Uninstalling tqdm-4.41.1:
      Successfully uninstalled tqdm-4.41.1


In [3]:
API_KEY = 'cc0a3da7611cfc6098a7bd9db11b3ecf' # Please get your your API Key from [https://www.aicrowd.com/participants/me]
!aicrowd login --api-key $API_KEY

API Key valid
Saved API Key successfully!


In [4]:
# Downloading the Dataset
!mkdir data
!aicrowd dataset download --challenge emotion-detection -j 3 -o data

train.csv:   0% 0.00/2.30M [00:00<?, ?B/s]
train.csv: 100% 2.30M/2.30M [00:00<00:00, 2.49MB/s]
val.csv:   0% 0.00/262k [00:00<?, ?B/s]
test.csv: 100% 642k/642k [00:00<00:00, 947kB/s]
val.csv: 100% 262k/262k [00:00<00:00, 518kB/s]


# Download & Import Libraries

In [5]:
!pip install emoji

     |████████████████████████████████| 133kB 4.1MB/s 


In [66]:
import os
import re
import emoji
import time
import pandas as pd
import nltk
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import f1_score, accuracy_score
from xgboost import XGBClassifier

In [43]:
nltk.download('punkt')
nltk.download('stopwords')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

# Read Dataset

In [9]:
train_dataset = pd.read_csv("data/train.csv")
validation_dataset = pd.read_csv("data/val.csv")[1:]
test_dataset = pd.read_csv("data/test.csv")

train_dataset.head(20)

,text,label
0,takes no time to copy/paste a press release,0
1,You're delusional,1
2,Jazz fan here. I completely feel. Lindsay Mann...,0
3,ah i was also confused but i think they mean f...,0
4,Thank you so much. ♥️ that means a lot.,0
5,And I’ll be there!!!,0
6,There are some amazingly cringey compilations ...,0
7,Check the frame (FPS) limit option in the adva...,0
8,you made me think I was in the dbd subreddit w...,0
9,It was in your op.,0


# Process text

In [44]:
stops = set(stopwords.words('english'))
porter = nltk.PorterStemmer()

def pre_process(str):
    def rm_html_tags(str):
        html_prog = re.compile(r'<[^>]+>',re.S)
        return html_prog.sub('', str)

    def rm_html_escape_characters(str):
        pattern_str = r'&quot;|&amp;|&lt;|&gt;|&nbsp;|&#34;|&#38;|&#60;|&#62;|&#160;|&#20284;|&#30524;|&#26684|&#43;|&#20540|&#23612;'
        escape_characters_prog = re.compile(pattern_str, re.S)
        return escape_characters_prog.sub('', str)

    def rm_at_user(str):
        return re.sub(r'@[a-zA-Z_0-9]*', '', str)

    def rm_url(str):
        return re.sub(r'http[s]?:[/+]?[a-zA-Z0-9_\.\/]*', '', str)

    def rm_repeat_chars(str):
        return re.sub(r'(.)(\1){2,}', r'\1\1', str)

    def rm_hashtag_symbol(str):
        return re.sub(r'#', '', str)

    def rm_time(str):
        return re.sub(r'[0-9][0-9]:[0-9][0-9]', '', str)

    def rm_punctuation(str):
        return re.sub(r'[^\w\s]', ' ', str)

    def split_emojis(str):
        text_part = ''.join(c for c in str if c not in emoji.UNICODE_EMOJI)
        emoji_part = ' '.join(c for c in str if c in emoji.UNICODE_EMOJI)
        return text_part + ' ' + emoji_part
    
    # do not change the preprocessing order only if you know what you're doing 
    str = str.lower()
    str = rm_url(str)        
    str = rm_at_user(str)        
    str = rm_repeat_chars(str) 
    str = rm_hashtag_symbol(str)       
    str = rm_time(str)
    str = rm_punctuation(str)
    # str = emoji.demojize(str, delimiters=(' emoji_', ' '))
    str = split_emojis(str)

    try:
        str = nltk.tokenize.word_tokenize(str)
        try:
            str = [porter.stem(t) for t in str]
        except:
            pass
    except:
        pass

    words = [w for w in str if w and w not in stops]
    return ' '.join(words)

In [45]:
text = "takes no time to copy/paste a press release"
pre_process(text)

'take time copi past press releas'

In [46]:
start_time = time.time()
train_dataset['processed_text'] = train_dataset['text'].apply(
    lambda x: pre_process(x)
)
print("Elapsed time: %s seconds" % round(time.time() - start_time, 4))

train_dataset.head(20)

Elapsed time: 12.734 seconds


,text,label,processed_text
0,takes no time to copy/paste a press release,0,take time copi past press releas
1,You're delusional,1,delusion
2,Jazz fan here. I completely feel. Lindsay Mann...,0,jazz fan complet feel lindsay mann cousin ha v...
3,ah i was also confused but i think they mean f...,0,ah wa also confus think mean friend around age
4,Thank you so much. ♥️ that means a lot.,0,thank much mean lot
5,And I’ll be there!!!,0,
6,There are some amazingly cringey compilations ...,0,amazingli cringey compil terribl dialogu thi s...
7,Check the frame (FPS) limit option in the adva...,0,check frame fp limit option advanc graphic opt...
8,you made me think I was in the dbd subreddit w...,0,made think wa dbd subreddit statement idk whi
9,It was in your op.,0,wa op


In [50]:
start_time = time.time()
validation_dataset['processed_text'] = validation_dataset['text'].apply(
    lambda x: pre_process(x)
)
print("Elapsed time: %s seconds" % round(time.time() - start_time, 4))

validation_dataset.head(20)

Elapsed time: 1.4703 seconds


,text,label,processed_text
1,im still starving,1,im still starv
2,*Hey just noticed..* it's your **2nd Cakeday**...,0,hey notic 2nd cakeday slumbishop hug
3,They just did. Check out the sticky post.,0,check sticki post
4,"I hope so too, she deserves it.",0,hope deserv
5,is it dangerous to take a quick photo while st...,0,danger take quick photo stop red light
6,I’m in my second year. Still closeted. Still u...,1,second year still closet still uncomfort bodi ...
7,"Noted, I've been looking into that",0,note look
8,Thank you for saying that I just haven’t felt ...,0,thank say felt right sad constantli
9,"Screw the watch stuff, I wanna hear about the ...",1,screw watch stuff wan na hear porn
10,"Exactly....we need a car tunnel, and then let ...",0,exactli need car tunnel let old road allow tru...


# Create TF-IDF Features

In [52]:
print(train_dataset.shape)
print(validation_dataset.shape)
combined_dataset = pd.concat([train_dataset, validation_dataset])
print(combined_dataset.shape)

(31255, 3)
(3472, 3)
(34727, 3)


In [53]:
tfidf_vect = TfidfVectorizer(analyzer='word', token_pattern=r'\w{1,}', max_features=5000)
tfidf_vect.fit(combined_dataset['processed_text'])

TfidfVectorizer(analyzer='word', binary=False, decode_error='strict',
                dtype=<class 'numpy.float64'>, encoding='utf-8',
                input='content', lowercase=True, max_df=1.0, max_features=5000,
                min_df=1, ngram_range=(1, 1), norm='l2', preprocessor=None,
                smooth_idf=True, stop_words=None, strip_accents=None,
                sublinear_tf=False, token_pattern='\\w{1,}', tokenizer=None,
                use_idf=True, vocabulary=None)

In [54]:
start_time = time.time()
xtrain_tfidf =  tfidf_vect.transform(train_dataset['processed_text'])
xval_tfidf =  tfidf_vect.transform(validation_dataset['processed_text'])
print("Elapsed time: %s seconds" % round(time.time() - start_time, 4))

print(xtrain_tfidf.shape)
print(xval_tfidf.shape)

Elapsed time: 0.3024 seconds
(31255, 5000)
(3472, 5000)


# Train Model

In [55]:
train_dataset['label'].value_counts()

0    24718
1     6537
Name: label, dtype: int64

In [56]:
y_train = train_dataset['label'].values
y_val = validation_dataset['label'].values

In [58]:
start_time = time.time()

model = XGBClassifier(
    n_estimators=1000, subsample=0.8, colsample_bytree=0.8,
    objective='binary:logistic', scale_pos_weight=4,
    seed=27
)
eval_set = [(xval_tfidf, y_val)]

model.fit(xtrain_tfidf, y_train, early_stopping_rounds=50,
          eval_metric="auc", eval_set=eval_set, verbose=True)
print(f"Elapsed time: {round(time.time() - start_time, 4)} seconds")

[0]	validation_0-auc:0.580752
Will train until validation_0-auc hasn't improved in 50 rounds.
[1]	validation_0-auc:0.603177
[2]	validation_0-auc:0.616018
[3]	validation_0-auc:0.630068
[4]	validation_0-auc:0.630116
[5]	validation_0-auc:0.630124
[6]	validation_0-auc:0.635991
[7]	validation_0-auc:0.635917
[8]	validation_0-auc:0.640824
[9]	validation_0-auc:0.642049
[10]	validation_0-auc:0.642151
[11]	validation_0-auc:0.647737
[12]	validation_0-auc:0.646662
[13]	validation_0-auc:0.650357
[14]	validation_0-auc:0.650415
[15]	validation_0-auc:0.65044
[16]	validation_0-auc:0.659476
[17]	validation_0-auc:0.663896
[18]	validation_0-auc:0.6639
[19]	validation_0-auc:0.674468
[20]	validation_0-auc:0.674488
[21]	validation_0-auc:0.676465
[22]	validation_0-auc:0.680267
[23]	validation_0-auc:0.680256
[24]	validation_0-auc:0.680231
[25]	validation_0-auc:0.686295
[26]	validation_0-auc:0.692176
[27]	validation_0-auc:0.69432
[28]	validation_0-auc:0.694512
[29]	validation_0-auc:0.698297
[30]	validation_0-au

# Evaluate Performance

In [59]:
train_dataset['pred'] = model.predict(xtrain_tfidf)
validation_dataset['pred'] = model.predict(xval_tfidf)

In [61]:
# Getting F1 & Accuracy score of training predictions
f1 = f1_score(train_dataset['label'], train_dataset['pred'])
accuracy = accuracy_score(train_dataset['label'], train_dataset['pred'])

print(f"Validation F1 Score  : {round(f1, 4)} and Accuracy Score {round(accuracy, 4)}")

Validation F1 Score  : 0.6398 and Accuracy Score 0.8345


In [62]:
# Getting F1 & Accuracy score of training predictions
f1 = f1_score(validation_dataset['label'], validation_dataset['pred'])
accuracy = accuracy_score(validation_dataset['label'], validation_dataset['pred'])

print(f"Validation F1 Score  : {round(f1, 4)} and Accuracy Score {round(accuracy, 4)}")

Validation F1 Score  : 0.5674 and Accuracy Score 0.7984


# Submit Results

In [63]:
test_dataset['processed_text'] = test_dataset['text'].apply(
    lambda x: pre_process(x)
)
xtest_tfidf =  tfidf_vect.transform(test_dataset['processed_text'])

print(xtest_tfidf.shape)

(8682, 5000)


In [64]:
test_dataset['label'] = model.predict(xtest_tfidf)
test_dataset.head()

,text,label,processed_text
0,I was already over the edge with Cassie Zamora...,1,wa alreadi edg cassi zamora show disdain two t...
1,I think you're right. She has oodles of cash a...,0,think right ha oodl cash young grandchildren e...
2,Haha I love this. I used to give mine phone bo...,0,haha love thi use give mine phone book room wo...
3,Probably out of desperation as they going no a...,0,probabl desper go answer made god
4,Sorry !! You’re real good at that!!,1,sorri real good


In [67]:
!mkdir assets

# Saving the sample submission in assets directory
if 'processed_text' in test_dataset.columns:
    test_dataset.drop(columns=['processed_text'], inplace=True)

test_dataset.to_csv(os.path.join("assets", "submission.csv"), index=False)

mkdir: cannot create directory ‘assets’: File exists


In [68]:
!aicrowd notebook submit -c emotion-detection -a assets --no-verify

Mounting Google Drive 💾
Your Google Drive will be mounted to access the colab notebook
Go to this URL in a browser: https://accounts.google.com/o/oauth2/auth?client_id=947318989803-6bn6qk8qdgf4n4g3pfee6491hc0brc4i.apps.googleusercontent.com&redirect_uri=urn%3aietf%3awg%3aoauth%3a2.0%3aoob&scope=email%20https%3a%2f%2fwww.googleapis.com%2fauth%2fdocs.test%20https%3a%2f%2fwww.googleapis.com%2fauth%2fdrive%20https%3a%2f%2fwww.googleapis.com%2fauth%2fdrive.photos.readonly%20https%3a%2f%2fwww.googleapis.com%2fauth%2fpeopleapi.readonly%20https%3a%2f%2fwww.googleapis.com%2fauth%2fdrive.activity.readonly%20https%3a%2f%2fwww.googleapis.com%2fauth%2fexperimentsandconfigs%20https%3a%2f%2fwww.googleapis.com%2fauth%2fphotos.native&response_type=code

Enter your authorization code:
4/1AY0e-g6VNJ4fp4Dnk3t4pje_ufmUZ0OEBQamTw_wGJJDRg5WBEelO3cSPBo
Mounted at /content/drive
Using notebook: /content/drive/MyDrive/Colab Notebooks/tfidf-xgboost-classifier.ipynb for submission...
Scrubbing API keys from the n